In [ ]:
# ============ TABLEAU DE BORD DE PILOTAGE – SIMPLON SOLUTION (version une seule cellule) ============
# Installation (une fois, dans une autre cellule) : !pip install pandas plotly ipywidgets openpyxl anywidget
import warnings
warnings.filterwarnings("ignore")


# ======================================================================
# 1. CONFIGURATION
# ======================================================================
from pathlib import Path

LOGO = "logo_simplon.png"          # logo Simplon Solutions (blanc sur fond rouge)

# Fichiers exportés depuis vos Google Sheets (CSV ou .xlsx)
FICHIER_RH      = "Volet RH & Effectifs.csv"
FICHIER_PROJETS = "SUIVI PROJET.csv"      # onglet « SUIVI PROJET » ou « projet »
FICHIER_AO      = "APPEL D'OFFRES.csv"

# --- Source DYNAMIQUE : Google Sheets (lecture en direct) ---
# Dans chaque Google Sheet : Partager > « Tous les utilisateurs disposant du lien » > Lecteur.
# Collez ici le lien complet (ou seulement l'identifiant) de chaque fichier.
SHEET_PILOTAGE = "https://docs.google.com/spreadsheets/d/1sPeiBYLY2WJAlENyLfsU9zcj-nK8FTRqd9H926tsRSA/edit?gid=0#gid=0"   # fichier « Pilotage – Simplon Solution » (onglet RH & Effectifs)
SHEET_SUIVI    = "https://docs.google.com/spreadsheets/d/18sZppjkDXyrn2eXMWqTH05QWwdyqab2IFYMTngyDYSU/edit?gid=772463690#gid=772463690"   # fichier contenant l'onglet « SUIVI PROJET » (ex. « SUIVI PROJET SS »)
SHEET_AO       = "https://docs.google.com/spreadsheets/d/18sZppjkDXyrn2eXMWqTH05QWwdyqab2IFYMTngyDYSU/edit?gid=1826812305#gid=1826812305"   # fichier contenant l'onglet « APPEL D'OFFRES » ; laissez vide s'il est dans le même fichier que SUIVI PROJET

ONGLET_RH      = "Volet RH & Effectifs"
ONGLET_PROJETS = "SUIVI PROJET"      # ou "projet"
ONGLET_AO      = "APPEL D'OFFRES"

ACTUALISATION_AUTO_SECONDES = 60     # délai de l'actualisation automatique (case « Auto »)

# Charte graphique Simplon : rouge, blanc, gris
ROUGE = "#E30047"      # rouge Simplon
BLANC = "#FFFFFF"
GRIS_CLAIR = "#CFD7DA" # gris Simplon
GRIS  = "#3A4047"      # gris foncé (textes)
PALETTE = [ROUGE, "#4A5258", "#9AA5AB", "#F2668F", "#2B3035", GRIS_CLAIR]  # rouge / gris / blanc


# ======================================================================
# 2. CHARGEMENT ET NETTOYAGE DES DONNÉES
# ======================================================================
import base64, re
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as W
from IPython.display import display, HTML, clear_output

pd.options.display.max_columns = 50

def lire(fichier):
    p = Path(fichier)
    if not p.exists():
        return None
    if p.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(p)
    for sep in (",", ";", "\t"):
        try:
            df = pd.read_csv(p, sep=sep, encoding="utf-8")
            if df.shape[1] > 2:
                return df
        except Exception:
            pass
    return pd.read_csv(p, sep=None, engine="python")

def norm(c):
    c = str(c).strip().lower()
    c = (c.replace("é","e").replace("è","e").replace("ê","e").replace("à","a")
           .replace("ç","c").replace("'","").replace("\n"," "))
    return re.sub(r"\s+", " ", c)

def renommer(df, mapping):
    """Renomme les colonnes selon des mots-clés (insensible à la casse/accents)."""
    out = {}
    for col in df.columns:
        n = norm(col)
        for cible, mots in mapping.items():
            if cible not in out.values() and any(m in n for m in mots):
                out[col] = cible
                break
    return df.rename(columns=out)

# --- Données de démonstration (utilisées seulement si les fichiers sont absents) ---
def demo_rh():
    return pd.DataFrame({
        "Prenom": list("ABCDEFGHIJ"), "Nom": [f"Nom{i}" for i in range(10)],
        "Role": ["Dev Web","Dev Web","Data","Designer","Chef de projet","Dev Mobile","Data","Dev Web","Designer","Support"],
        "Statut": ["CDI","CDD","Stagiaire","CDI","CDI","Alternant","CDD","CDI","Stagiaire","CDI"],
        "Fin contrat": pd.to_datetime(["2027-03-01","2026-12-15","2026-12-31","2027-06-30","2028-01-01",
                                       "2027-08-31","2026-11-30","2027-02-28","2026-12-20","2028-05-01"]),
        "Experience": ["Junior","Confirmé","Junior","Senior","Senior","Junior","Confirmé","Confirmé","Junior","Senior"],
        "Stack": ["PHP/Laravel","React","Python","Figma","Gestion","Flutter","Python","WordPress","Figma","Réseau"],
    })

def demo_projets():
    rng = np.random.default_rng(1)
    n = 14
    return pd.DataFrame({
        "Projet": [f"Projet {i+1}" for i in range(n)],
        "Etat": rng.choice(["En cours","Terminé","En attente"], n),
        "Debut": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 700, n), unit="D"),
        "Delais": rng.integers(15, 120, n),
        "Categorie": rng.choice(["Interne","Externe"], n),
        "Typologie client": rng.choice(["Entreprise","Institution","ONG","Particulier"], n),
        "Livrable": rng.choice(["Site Web","Application","Plateforme","Identité visuelle"], n),
        "Annee": rng.choice([2023, 2024, 2025], n),
    })

def demo_ao():
    rng = np.random.default_rng(2)
    n = 20
    return pd.DataFrame({
        "Entreprise": [f"Client {i+1}" for i in range(n)],
        "Date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 700, n), unit="D"),
        "Action": rng.choice(["Dépôt en ligne","Dépôt physique","Via plateforme"], n),
        "Services": rng.choice(["Site Web","Application","Maintenance","Hébergement"], n),
        "Resultat": rng.choice(["PERDUS","EN COURS","GAGNER"], n, p=[.5,.25,.25]),
        "Annee": rng.choice([2024, 2025, 2026], n),
    })

MAP_RH = {"Prenom":["prenom"], "Nom":["nom"], "Role":["role"], "Statut":["statut"],
          "Fin contrat":["fin de contract","fin de contrat","fin contrat"],
          "Experience":["experience"], "Stack":["stack"]}
MAP_PROJ = {"Projet":["nom projet","projet"], "Etat":["etat"], "Debut":["debut"], "Delais":["delai"],
            "Commentaire":["commentaire"], "Personnes":["personnes"], "Categorie":["categorie"],
            "Typologie client":["typologie"], "Livrable":["livrable"], "Annee":["annee"]}
MAP_AO = {"Entreprise":["entreprise"], "Date":["date"], "Action":["action"], "Services":["services"],
          "Resultat":["resultat"], "Commentaires":["commentaire"], "Offres":["offres"], "Annee":["annee"]}

import io, urllib.parse, urllib.request
from datetime import datetime

def sheet_id(x):
    m = re.search(r"/d/([a-zA-Z0-9_-]+)", x or "")
    return m.group(1) if m else (x or "").strip()

def lire_google(source, onglet):
    """Lit un onglet Google Sheets en direct (le fichier doit être partagé en lecture par lien)."""
    sid = sheet_id(source)
    if not sid:
        return None
    url = ("https://docs.google.com/spreadsheets/d/" + sid +
           "/gviz/tq?tqx=out:csv&sheet=" + urllib.parse.quote(onglet))
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            contenu = r.read()
        if contenu.lstrip()[:1] == b"<":
            raise ValueError("accès refusé : vérifiez le partage « Tous les utilisateurs disposant du lien »")
        return pd.read_csv(io.BytesIO(contenu))
    except Exception as e:
        print("⚠️ Google Sheets - onglet « " + onglet + " » : " + str(e))
        return None

SOURCE = {}
DERNIERE_MAJ = None
rh = projets = ao = None

def charger():
    """(Re)charge les 3 jeux de données : Google Sheets > fichier local > démonstration."""
    global rh, projets, ao, DERNIERE_MAJ
    jeux = {
        "RH":              (SHEET_PILOTAGE, ONGLET_RH,      FICHIER_RH,      MAP_RH,   demo_rh),
        "Projets":         (SHEET_SUIVI,    ONGLET_PROJETS, FICHIER_PROJETS, MAP_PROJ, demo_projets),
        "Appels d'offres": (SHEET_AO or SHEET_SUIVI, ONGLET_AO,      FICHIER_AO,      MAP_AO,   demo_ao),
    }
    res = {}
    for nom, (src, onglet, fichier, mp, demo_f) in jeux.items():
        df, origine = lire_google(src, onglet), "Google Sheets"
        if df is None:
            df, origine = lire(fichier), "fichier local"
        if df is None:
            df, origine = demo_f(), "démonstration"
        else:
            df = renommer(df, mp)
        res[nom], SOURCE[nom] = df, origine
    r, p, a = res["RH"], res["Projets"], res["Appels d'offres"]

    for df, cols in ((r, ["Fin contrat"]), (p, ["Debut"]), (a, ["Date"])):
        for c in cols:
            if c in df: df[c] = pd.to_datetime(df[c], errors="coerce", dayfirst=True)
    for df in (p, a):
        if "Annee" in df: df["Annee"] = pd.to_numeric(df["Annee"], errors="coerce").astype("Int64")
    if "Resultat" in a: a["Resultat"] = a["Resultat"].astype(str).str.strip().str.upper()
    if "Etat" in p:     p["Etat"] = p["Etat"].astype(str).str.strip().str.capitalize()

    rh, projets, ao = r, p, a
    DERNIERE_MAJ = datetime.now()

charger()


# ======================================================================
# 3. OUTILS D'AFFICHAGE
# ======================================================================
def logo_html(hauteur=64):
    p = Path(LOGO)
    if not p.exists():
        return f'<div style="font-weight:800;font-size:22px;color:{BLANC}">simplon SOLUTIONS</div>'
    mime = {"png":"image/png","jpg":"image/jpeg","jpeg":"image/jpeg","svg":"image/svg+xml"}.get(p.suffix[1:].lower(), "image/png")
    b64 = base64.b64encode(p.read_bytes()).decode()
    return f'<img src="data:{mime};base64,{b64}" style="height:{hauteur}px">'

def entete(titre, sous_titre=""):
    return HTML(f"""
    <div style="display:flex;align-items:center;justify-content:space-between;background:{ROUGE};
                border-radius:8px;padding:10px 22px;margin-bottom:14px;font-family:Segoe UI,Arial,sans-serif">
      <div><div style="font-size:24px;font-weight:700;color:{BLANC}">{titre}</div>
           <div style="font-size:13px;color:{BLANC};opacity:.85">{sous_titre}</div></div>
      {logo_html()}
    </div>""")

def kpi(cartes):
    html = "".join(f"""
      <div style="flex:1;min-width:150px;background:{BLANC};border-left:5px solid {ROUGE};
                  box-shadow:0 1px 4px rgba(0,0,0,.18);border-radius:6px;padding:12px 16px">
        <div style="font-size:12px;color:#6B747A;text-transform:uppercase">{lbl}</div>
        <div style="font-size:28px;font-weight:700;color:{GRIS}">{val}</div></div>""" for lbl, val in cartes)
    return HTML(f'<div style="display:flex;gap:12px;flex-wrap:wrap;margin:8px 0 14px;font-family:Segoe UI,Arial">{html}</div>')

def style(fig, titre, h=340):
    fig.update_layout(title=titre, height=h, margin=dict(l=20, r=20, t=50, b=20),
                      template="plotly_white", paper_bgcolor=BLANC, plot_bgcolor=BLANC, font=dict(color=GRIS), colorway=PALETTE, legend_title_text="")
    return fig

def camembert(df, col, titre):
    d = df[col].fillna("Non renseigné").value_counts().reset_index()
    d.columns = [col, "n"]
    return style(px.pie(d, names=col, values="n", hole=.45), titre)

def barres(df, col, titre, horizontal=False):
    d = df[col].fillna("Non renseigné").value_counts().reset_index()
    d.columns = [col, "n"]
    fig = px.bar(d, x="n" if horizontal else col, y=col if horizontal else "n",
                 orientation="h" if horizontal else "v", text="n", color_discrete_sequence=[ROUGE])
    fig.update_traces(textposition="outside")
    return style(fig, titre)

def filtre_options(df, col):
    return ["Tous"] + sorted(df[col].dropna().astype(str).unique()) if col in df else ["Tous"]

def appliquer(df, col, val):
    return df if (val == "Tous" or col not in df) else df[df[col].astype(str) == val]

AIDE = ("<div style='font-family:Segoe UI,Arial;font-size:12px;color:#6B747A;margin:4px 0 8px'>"
        "💡 Cliquez sur une part ou une barre d'un graphique pour filtrer toute la page ; "
        "cliquez à nouveau dessus pour revenir à « Tous ».</div>")

def creer_filtres(df, specs):
    return {col: W.Dropdown(options=filtre_options(df, col), description=lbl,
                            layout=W.Layout(width="250px")) for col, lbl in specs}

def filtrer(df, filtres):
    for col, w in filtres.items():
        df = appliquer(df, col, w.value)
    return df

def afficher(fig, dropdown=None, horizontal=False):
    """Affiche un graphique. Avec un filtre associé : un clic sur une part/barre filtre la page."""
    if dropdown is None:
        fig.show()
        return
    try:
        fw = go.FigureWidget(fig)

        def au_clic(trace, points, state):
            if not points.point_inds:
                return
            i = points.point_inds[0]
            if trace.type == "pie":
                val = trace.labels[i]
            else:
                val = trace.y[i] if horizontal else trace.x[i]
            val = str(val)
            if val in dropdown.options:
                dropdown.value = "Tous" if dropdown.value == val else val

        for t in fw.data:
            t.on_click(au_clic)
        display(fw)
    except Exception:
        fig.show()   # repli : graphique non cliquable si les widgets Plotly ne sont pas disponibles


# ======================================================================
# 4. PAGE RH & EFFECTIFS
# ======================================================================
def page_rh():
    f = creer_filtres(rh, [("Statut", "Statut :"), ("Experience", "Expérience :"),
                           ("Role", "Rôle :"), ("Stack", "Stack :")])
    out = W.Output()

    def rendre(*_):
        d = filtrer(rh, f)
        with out:
            clear_output(wait=True)
            prochains = 0
            if "Fin contrat" in d:
                limite = pd.Timestamp.today() + pd.Timedelta(days=90)
                prochains = int(((d["Fin contrat"] <= limite) & (d["Fin contrat"] >= pd.Timestamp.today())).sum())
            display(kpi([("Effectif", len(d)),
                         ("Rôles distincts", d["Role"].nunique() if "Role" in d else "–"),
                         ("Stacks distinctes", d["Stack"].nunique() if "Stack" in d else "–"),
                         ("Fins de contrat < 90 j", prochains)]))
            if "Statut" in d:     afficher(camembert(d, "Statut", "Répartition par statut"), f["Statut"])
            if "Experience" in d: afficher(barres(d, "Experience", "Niveau d'expérience"), f["Experience"])
            if "Stack" in d:      afficher(barres(d, "Stack", "Répartition par stack", horizontal=True), f["Stack"], True)
            if "Role" in d:       afficher(barres(d, "Role", "Répartition par rôle", horizontal=True), f["Role"], True)
            if "Fin contrat" in d:
                cols = [c for c in ["Prenom", "Nom", "Role", "Statut", "Fin contrat"] if c in d]
                display(HTML("<h4>Contrats à échéance proche</h4>"))
                display(d.sort_values("Fin contrat")[cols].head(10))
    for w in f.values(): w.observe(rendre, "value")
    rendre()
    return W.VBox([W.HTML(AIDE), W.HBox(list(f.values()), layout=W.Layout(flex_flow="row wrap")), out])


# ======================================================================
# 5. PAGE PROJETS & ACTIVITÉ
# ======================================================================
def page_projets():
    f = creer_filtres(projets, [("Annee", "Année :"), ("Categorie", "Catégorie :"),
                                ("Typologie client", "Client :"), ("Etat", "État :"),
                                ("Livrable", "Livrable :")])
    out = W.Output()

    def rendre(*_):
        d = filtrer(projets, f)
        with out:
            clear_output(wait=True)
            en_cours = int(d["Etat"].str.lower().str.contains("cours", na=False).sum()) if "Etat" in d else "–"
            termines = int(d["Etat"].str.lower().str.contains("termin", na=False).sum()) if "Etat" in d else "–"
            delai = round(pd.to_numeric(d["Delais"], errors="coerce").mean(), 1) if "Delais" in d else "–"
            display(kpi([("Projets", len(d)), ("En cours", en_cours), ("Terminés", termines),
                         ("Délai moyen (j)", delai)]))
            if "Etat" in d:             afficher(camembert(d, "Etat", "Projets par état"), f["Etat"])
            if "Typologie client" in d: afficher(camembert(d, "Typologie client", "Typologie de client"), f["Typologie client"])
            if "Livrable" in d:         afficher(barres(d, "Livrable", "Projets par type de livrable"), f["Livrable"])
            if "Annee" in d:
                a = d.groupby("Annee").size().reset_index(name="n")
                fig = style(px.bar(a, x="Annee", y="n", text="n", color_discrete_sequence=[ROUGE]), "Projets par année")
                fig.update_xaxes(type="category")
                afficher(fig, f["Annee"])
            cols = [c for c in ["Projet", "Etat", "Debut", "Delais", "Categorie", "Typologie client", "Livrable", "Annee"] if c in d]
            display(HTML("<h4>Détail des projets</h4>")); display(d[cols])
    for w in f.values(): w.observe(rendre, "value")
    rendre()
    return W.VBox([W.HTML(AIDE), W.HBox(list(f.values()), layout=W.Layout(flex_flow="row wrap")), out])


# ======================================================================
# 6. PAGE APPELS D'OFFRES
# ======================================================================
def page_ao():
    f = creer_filtres(ao, [("Annee", "Année :"), ("Resultat", "Résultat :"),
                           ("Action", "Dépôt :"), ("Services", "Service :")])
    out = W.Output()

    def rendre(*_):
        d = filtrer(ao, f)
        with out:
            clear_output(wait=True)
            tot = len(d)
            gagnes = int((d["Resultat"] == "GAGNER").sum()) if "Resultat" in d else 0
            perdus = int((d["Resultat"] == "PERDUS").sum()) if "Resultat" in d else 0
            cours  = int((d["Resultat"] == "EN COURS").sum()) if "Resultat" in d else 0
            taux = f"{gagnes / (gagnes + perdus) * 100:.0f} %" if (gagnes + perdus) else "–"
            display(kpi([("Appels d'offres", tot), ("Gagnés", gagnes), ("Perdus", perdus),
                         ("En cours", cours), ("Taux de réussite", taux)]))
            if "Resultat" in d: afficher(camembert(d, "Resultat", "Résultats"), f["Resultat"])
            if "Action" in d:   afficher(barres(d, "Action", "Mode de dépôt"), f["Action"])
            if "Services" in d: afficher(barres(d, "Services", "Services demandés", horizontal=True), f["Services"], True)
            if "Date" in d and d["Date"].notna().any():
                m = d.dropna(subset=["Date"]).assign(Mois=lambda x: x["Date"].dt.to_period("M").astype(str))
                m = m.groupby("Mois").size().reset_index(name="n")
                afficher(style(px.line(m, x="Mois", y="n", markers=True, color_discrete_sequence=[ROUGE]),
                               "Appels d'offres par mois"))
            cols = [c for c in ["Entreprise", "Date", "Action", "Services", "Resultat", "Commentaires", "Annee"] if c in d]
            display(HTML("<h4>Détail</h4>")); display(d[cols])
    for w in f.values(): w.observe(rendre, "value")
    rendre()
    return W.VBox([W.HTML(AIDE), W.HBox(list(f.values()), layout=W.Layout(flex_flow="row wrap")), out])


# ======================================================================
# 7. ACCUEIL ET ASSEMBLAGE DU TABLEAU DE BORD
# ======================================================================
def page_accueil():
    out = W.Output()
    with out:
        taux = "–"
        if "Resultat" in ao:
            g, p = (ao["Resultat"] == "GAGNER").sum(), (ao["Resultat"] == "PERDUS").sum()
            taux = f"{g / (g + p) * 100:.0f} %" if (g + p) else "–"
        en_cours = int(projets["Etat"].str.lower().str.contains("cours", na=False).sum()) if "Etat" in projets else "–"
        display(kpi([("Effectif", len(rh)), ("Projets", len(projets)), ("Projets en cours", en_cours),
                     ("Appels d'offres", len(ao)), ("Taux de réussite AO", taux)]))
        display(HTML("<p style='font-family:Segoe UI,Arial;color:#555'>Utilisez les onglets ci-dessus pour explorer "
                     "<b>RH &amp; effectifs</b>, <b>Projets &amp; activité</b> et <b>Appels d'offres</b>.</p>"))
    return out

import asyncio

display(entete("Tableau de bord de pilotage", "Simplon Solution – RH, projets et appels d'offres"))

zone   = W.Output()
statut = W.HTML()
bouton = W.Button(description="Actualiser les données", icon="refresh", button_style="danger",
                  layout=W.Layout(width="220px"))
auto   = W.Checkbox(value=False, indent=False,
                    description="Actualisation auto (" + str(ACTUALISATION_AUTO_SECONDES) + " s)")
etat   = {"onglet": 0, "tache": None}
TITRES = ["Accueil", "RH & effectifs", "Projets & activité", "Appels d'offres"]

def construire():
    tabs = W.Tab(children=[page_accueil(), page_rh(), page_projets(), page_ao()])
    for i, t in enumerate(TITRES): tabs.set_title(i, t)
    tabs.selected_index = etat["onglet"]
    tabs.observe(lambda ch: etat.update(onglet=ch["new"]), "selected_index")
    with zone:
        clear_output(wait=True)
        display(tabs)
    statut.value = ("<span style='font-family:Segoe UI,Arial;font-size:12px;color:" + GRIS + "'>"
                    "Dernière mise à jour : <b>" + DERNIERE_MAJ.strftime("%d/%m/%Y %H:%M:%S") + "</b>"
                    "</span>")

def actualiser(*_):
    bouton.disabled = True
    bouton.description = "Mise à jour…"
    try:
        charger()
        construire()
    finally:
        bouton.description = "Actualiser les données"
        bouton.disabled = False

async def boucle_auto():
    while auto.value:
        await asyncio.sleep(ACTUALISATION_AUTO_SECONDES)
        if auto.value:
            actualiser()

def basculer_auto(ch):
    if ch["new"]:
        etat["tache"] = asyncio.ensure_future(boucle_auto())
    elif etat["tache"] is not None:
        etat["tache"].cancel()

bouton.on_click(actualiser)
auto.observe(basculer_auto, "value")

display(W.HBox([bouton, auto, statut], layout=W.Layout(align_items="center", gap="16px")))
display(zone)
construire()
